# Exploratory Data Analysis for NBA Lineup Prediction and Hidden Patterns

This notebook organizes the data exploration steps from the raw CSV file and then implements several additional views to uncover hidden patterns. We aim to understand:

- **Lineup stability and variation** over time
- **Frequency of player appearances** in different positions (home_0 to home_4)
- **Outcome analysis** by lineup
- **Common matchups:** which home players appear most frequently against certain away lineups

These insights can guide further feature engineering to improve our predictive model.

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns

# For warnings
import warnings
warnings.filterwarnings('ignore')

# Set display options for Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

pd.option_context('display.max_rows', None, 'display.max_columns', None)

## 1. Load and Inspect the Raw Data

Here we load a sample raw CSV file (2007 season) and inspect its contents.

In [9]:
# Define the folder containing the CSV files (update the path as needed)
data_folder = "../data"  # Replace with your actual data folder path

# Load all CSV files from 2007 to 2015
csv_files = sorted(glob.glob(os.path.join(data_folder, "matchups-20*.csv")))
dfs = []
for file in csv_files:
    print(f"Loading {file}")
    df = pd.read_csv(file)
    dfs.append(df)

df_all_years = pd.concat(dfs, ignore_index=True)

print("Dataset loaded with shape:", df_all_years.shape)
display(df_all_years.head())

Loading ../data\matchups-2007.csv
Loading ../data\matchups-2008.csv
Loading ../data\matchups-2009.csv
Loading ../data\matchups-2010.csv
Loading ../data\matchups-2011.csv
Loading ../data\matchups-2012.csv
Loading ../data\matchups-2013.csv
Loading ../data\matchups-2014.csv
Loading ../data\matchups-2015.csv
Dataset loaded with shape: (236912, 53)


,game,season,home_team,away_team,starting_min,end_min,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4,fga_home,fta_home,fgm_home,fga_2_home,fgm_2_home,fga_3_home,fgm_3_home,ast_home,blk_home,pf_home,reb_home,dreb_home,oreb_home,to_home,pts_home,pct_home,pct_2_home,pct_3_home,fga_visitor,fta_visitor,fgm_visitor,fga_2_visitor,fgm_2_visitor,fga_3_visitor,fgm_3_visitor,ast_visitor,blk_visitor,pf_visitor,reb_visitor,dreb_visitor,oreb_visitor,to_visitor,pts_visitor,pct_visitor,pct_2_visitor,pct_3_visitor,outcome
0,200610310LAL,2007,LAL,PHO,0,5,Andrew Bynum,Lamar Odom,Luke Walton,Sasha Vujacic,Smush Parker,Boris Diaw,Kurt Thomas,Raja Bell,Shawn Marion,Steve Nash,10,4,4,7,3,3,1,3,0,1,1,1,0,1,9,0.400000,0.428571,0.333333,11,1,10,8,8,3,2,6,0,0,7,7,0,1,22,0.909091,1.00,0.666667,-1
1,200610310LAL,2007,LAL,PHO,6,7,Andrew Bynum,Lamar Odom,Luke Walton,Sasha Vujacic,Smush Parker,Amar'e Stoudemire,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,3,0,2,2,2,1,0,1,0,0,2,2,0,2,4,0.666667,1.000000,0.000000,6,0,4,4,3,2,1,4,0,0,1,1,0,0,9,0.666667,0.75,0.500000,-1
2,200610310LAL,2007,LAL,PHO,8,9,Lamar Odom,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Amar'e Stoudemire,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,4,0,2,4,2,0,0,2,1,1,2,1,1,2,4,0.500000,0.500000,0.000000,2,2,1,2,1,0,0,1,0,0,1,1,0,1,2,0.500000,0.50,0.000000,1
3,200610310LAL,2007,LAL,PHO,10,10,Lamar Odom,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Boris Diaw,James Jones,Kurt Thomas,Leandro Barbosa,Marcus Banks,2,0,2,2,2,0,0,1,0,0,0,0,0,0,4,1.000000,1.000000,0.000000,2,0,1,0,0,2,1,1,0,0,1,0,1,1,3,0.500000,0.00,0.500000,1
4,200610310LAL,2007,LAL,PHO,11,11,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Vladimir Radmanovic,Boris Diaw,James Jones,Kurt Thomas,Leandro Barbosa,Marcus Banks,2,0,1,2,1,0,0,1,0,1,0,0,0,0,2,0.500000,0.500000,0.000000,1,0,1,1,1,0,0,1,0,1,1,1,0,1,2,1.000000,1.00,0.000000,-1


## 2. Filter to Allowed Features and Create a Date Column

We will filter the data to include only the allowed features and then extract a proper date from the `game` column.

In [78]:
# Define the allowed features
allowed_features = [
    'game', 'season', 'home_team', 'away_team', 'starting_min',
    'home_0', 'home_1', 'home_2', 'home_3', 'home_4',
    'away_0', 'away_1', 'away_2', 'away_3', 'away_4',
    'outcome'  
]

# Filter the dataframe
df_filtered = df_all_years[allowed_features].copy()

# Create a date column from the first 8 characters of the 'game' column
df_filtered['date'] = pd.to_datetime(df_filtered['game'].str[:8], format='%Y%m%d')

# 🔹 Replace "CHO" with "CHA" in the home_team column
df_filtered['home_team'] = df_filtered['home_team'].replace('CHO', 'CHA')
df_filtered['away_team'] = df_filtered['home_team'].replace('CHO', 'CHA')

df_filtered

,game,season,home_team,away_team,starting_min,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4,outcome,date
0,200610310LAL,2007,LAL,LAL,0,Andrew Bynum,Lamar Odom,Luke Walton,Sasha Vujacic,Smush Parker,Boris Diaw,Kurt Thomas,Raja Bell,Shawn Marion,Steve Nash,-1,2006-10-31
1,200610310LAL,2007,LAL,LAL,6,Andrew Bynum,Lamar Odom,Luke Walton,Sasha Vujacic,Smush Parker,Amar'e Stoudemire,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,-1,2006-10-31
2,200610310LAL,2007,LAL,LAL,8,Lamar Odom,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Amar'e Stoudemire,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,1,2006-10-31
3,200610310LAL,2007,LAL,LAL,10,Lamar Odom,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Boris Diaw,James Jones,Kurt Thomas,Leandro Barbosa,Marcus Banks,1,2006-10-31
4,200610310LAL,2007,LAL,LAL,11,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Vladimir Radmanovic,Boris Diaw,James Jones,Kurt Thomas,Leandro Barbosa,Marcus Banks,-1,2006-10-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236907,201504050NYK,2015,NYK,NYK,35,Jason Smith,Quincy Acy,Ricky Ledo,Shane Larkin,Tim Hardaway,Henry Sims,Hollis Thompson,JaKarr Sampson,Jason Richardson,Jerami Grant,-1,2015-04-05
236908,201504050NYK,2015,NYK,NYK,39,Jason Smith,Langston Galloway,Quincy Acy,Ricky Ledo,Tim Hardaway,Henry Sims,Hollis Thompson,JaKarr Sampson,Jason Richardson,Jerami Grant,-1,2015-04-05
236909,201504050NYK,2015,NYK,NYK,40,Jason Smith,Langston Galloway,Quincy Acy,Ricky Ledo,Tim Hardaway,Furkan Aldemir,Hollis Thompson,JaKarr Sampson,Jason Richardson,Nerlens Noel,-1,2015-04-05
236910,201504050NYK,2015,NYK,NYK,42,Andrea Bargnani,Jason Smith,Lance Thomas,Langston Galloway,Shane Larkin,Furkan Aldemir,Ish Smith,Jerami Grant,Nerlens Noel,Robert Covington,-1,2015-04-05


## 3. Create Lineup Tuples for Home and Away Teams

To explore lineup frequency and uniqueness, we create sorted tuples of players for both the home and away teams.

In [79]:
# Create lineup tuples (sorted alphabetically for consistency)
df_filtered['home_lineup'] = df_filtered[['home_0', 'home_1', 'home_2', 'home_3', 'home_4']].apply(lambda x: tuple(sorted(x)), axis=1)
df_filtered['away_lineup'] = df_filtered[['away_0', 'away_1', 'away_2', 'away_3', 'away_4']].apply(lambda x: tuple(sorted(x)), axis=1)

df_filtered[['game', 'home_lineup', 'away_lineup', 'date']]

,game,home_lineup,away_lineup,date
0,200610310LAL,"(Andrew Bynum, Lamar Odom, Luke Walton, Sasha Vujacic, Smush Parker)","(Boris Diaw, Kurt Thomas, Raja Bell, Shawn Marion, Steve Nash)",2006-10-31
1,200610310LAL,"(Andrew Bynum, Lamar Odom, Luke Walton, Sasha Vujacic, Smush Parker)","(Amar'e Stoudemire, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)",2006-10-31
2,200610310LAL,"(Lamar Odom, Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker)","(Amar'e Stoudemire, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)",2006-10-31
3,200610310LAL,"(Lamar Odom, Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker)","(Boris Diaw, James Jones, Kurt Thomas, Leandro Barbosa, Marcus Banks)",2006-10-31
4,200610310LAL,"(Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker, Vladimir Radmanovic)","(Boris Diaw, James Jones, Kurt Thomas, Leandro Barbosa, Marcus Banks)",2006-10-31
...,...,...,...,...
236907,201504050NYK,"(Jason Smith, Quincy Acy, Ricky Ledo, Shane Larkin, Tim Hardaway)","(Henry Sims, Hollis Thompson, JaKarr Sampson, Jason Richardson, Jerami Grant)",2015-04-05
236908,201504050NYK,"(Jason Smith, Langston Galloway, Quincy Acy, Ricky Ledo, Tim Hardaway)","(Henry Sims, Hollis Thompson, JaKarr Sampson, Jason Richardson, Jerami Grant)",2015-04-05
236909,201504050NYK,"(Jason Smith, Langston Galloway, Quincy Acy, Ricky Ledo, Tim Hardaway)","(Furkan Aldemir, Hollis Thompson, JaKarr Sampson, Jason Richardson, Nerlens Noel)",2015-04-05
236910,201504050NYK,"(Andrea Bargnani, Jason Smith, Lance Thomas, Langston Galloway, Shane Larkin)","(Furkan Aldemir, Ish Smith, Jerami Grant, Nerlens Noel, Robert Covington)",2015-04-05


## 4. Unique Lineup Counts per Game

We group by game and count how many unique home and away lineups were used.

In [80]:
# Count unique lineups per game
lineup_counts = df_filtered.groupby('game').agg(
    unique_home_lineups=('home_lineup', 'nunique'),
    unique_away_lineups=('away_lineup', 'nunique')
).reset_index()

lineup_counts

,game,unique_home_lineups,unique_away_lineups
0,200610310LAL,11,9
1,200610310MIA,11,16
2,200611010BOS,19,17
3,200611010CHA,16,15
4,200611010CLE,12,10
...,...,...,...
10823,201504150MIN,15,14
10824,201504150NOP,14,16
10825,201504150NYK,12,14
10826,201504150PHI,9,2


In [81]:
unique_games = df_all_years['game'].nunique()
print("Total unique games:", unique_games)

Total unique games: 10828


In [8]:
df_filtered

,game,season,home_team,away_team,starting_min,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4,outcome,date,home_lineup,away_lineup
0,200610310LAL,2007,LAL,PHO,0,Andrew Bynum,Lamar Odom,Luke Walton,Sasha Vujacic,Smush Parker,Boris Diaw,Kurt Thomas,Raja Bell,Shawn Marion,Steve Nash,-1,2006-10-31,"(Andrew Bynum, Lamar Odom, Luke Walton, Sasha Vujacic, Smush Parker)","(Boris Diaw, Kurt Thomas, Raja Bell, Shawn Marion, Steve Nash)"
1,200610310LAL,2007,LAL,PHO,6,Andrew Bynum,Lamar Odom,Luke Walton,Sasha Vujacic,Smush Parker,Amar'e Stoudemire,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,-1,2006-10-31,"(Andrew Bynum, Lamar Odom, Luke Walton, Sasha Vujacic, Smush Parker)","(Amar'e Stoudemire, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)"
2,200610310LAL,2007,LAL,PHO,8,Lamar Odom,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Amar'e Stoudemire,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,1,2006-10-31,"(Lamar Odom, Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker)","(Amar'e Stoudemire, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)"
3,200610310LAL,2007,LAL,PHO,10,Lamar Odom,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Boris Diaw,James Jones,Kurt Thomas,Leandro Barbosa,Marcus Banks,1,2006-10-31,"(Lamar Odom, Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker)","(Boris Diaw, James Jones, Kurt Thomas, Leandro Barbosa, Marcus Banks)"
4,200610310LAL,2007,LAL,PHO,11,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Vladimir Radmanovic,Boris Diaw,James Jones,Kurt Thomas,Leandro Barbosa,Marcus Banks,-1,2006-10-31,"(Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker, Vladimir Radmanovic)","(Boris Diaw, James Jones, Kurt Thomas, Leandro Barbosa, Marcus Banks)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236907,201504050NYK,2015,NYK,PHI,35,Jason Smith,Quincy Acy,Ricky Ledo,Shane Larkin,Tim Hardaway,Henry Sims,Hollis Thompson,JaKarr Sampson,Jason Richardson,Jerami Grant,-1,2015-04-05,"(Jason Smith, Quincy Acy, Ricky Ledo, Shane Larkin, Tim Hardaway)","(Henry Sims, Hollis Thompson, JaKarr Sampson, Jason Richardson, Jerami Grant)"
236908,201504050NYK,2015,NYK,PHI,39,Jason Smith,Langston Galloway,Quincy Acy,Ricky Ledo,Tim Hardaway,Henry Sims,Hollis Thompson,JaKarr Sampson,Jason Richardson,Jerami Grant,-1,2015-04-05,"(Jason Smith, Langston Galloway, Quincy Acy, Ricky Ledo, Tim Hardaway)","(Henry Sims, Hollis Thompson, JaKarr Sampson, Jason Richardson, Jerami Grant)"
236909,201504050NYK,2015,NYK,PHI,40,Jason Smith,Langston Galloway,Quincy Acy,Ricky Ledo,Tim Hardaway,Furkan Aldemir,Hollis Thompson,JaKarr Sampson,Jason Richardson,Nerlens Noel,-1,2015-04-05,"(Jason Smith, Langston Galloway, Quincy Acy, Ricky Ledo, Tim Hardaway)","(Furkan Aldemir, Hollis Thompson, JaKarr Sampson, Jason Richardson, Nerlens Noel)"
236910,201504050NYK,2015,NYK,PHI,42,Andrea Bargnani,Jason Smith,Lance Thomas,Langston Galloway,Shane Larkin,Furkan Aldemir,Ish Smith,Jerami Grant,Nerlens Noel,Robert Covington,-1,2015-04-05,"(Andrea Bargnani, Jason Smith, Lance Thomas, Langston Galloway, Shane Larkin)","(Furkan Aldemir, Ish Smith, Jerami Grant, Nerlens Noel, Robert Covington)"


## 5. Most Frequently Used Lineups per Game

We now identify the most frequently used home and away lineups for each game.

In [82]:
# Sort the DataFrame by game and starting_min
df_sorted = df_filtered.sort_values(['game', 'starting_min']).copy()

# Compute the duration for each lineup segment per game.
# For each game, duration = next starting_min - current starting_min.
# For the last segment in each game, duration = 48 - current starting_min.
df_sorted['duration'] = df_sorted.groupby('game')['starting_min'].transform(lambda x: x.shift(-1) - x)
df_sorted['duration'] = df_sorted['duration'].fillna(48 - df_sorted['starting_min'])

# Group by the lineup combination and sum the duration (total minutes played)
lineup_usage = df_sorted.groupby(
    ['game', 'home_team', 'away_team', 'date', 
     'home_0', 'home_1', 'home_2', 'home_3', 'home_4',
     'away_0', 'away_1', 'away_2', 'away_3', 'away_4', 
     'home_lineup', 'away_lineup']
)['duration'].sum().reset_index(name='total_minutes')

# Sort the DataFrame by game and total_minutes in descending order
most_used_lineups = lineup_usage.sort_values(['game', 'total_minutes'], ascending=[True, False])
display(most_used_lineups)

,game,home_team,away_team,date,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4,home_lineup,away_lineup,total_minutes
8,200610310LAL,LAL,LAL,2006-10-31,Brian Cook,Jordan Farmar,Lamar Odom,Sasha Vujacic,Vladimir Radmanovic,Boris Diaw,James Jones,Kurt Thomas,Leandro Barbosa,Marcus Banks,"(Brian Cook, Jordan Farmar, Lamar Odom, Sasha Vujacic, Vladimir Radmanovic)","(Boris Diaw, James Jones, Kurt Thomas, Leandro Barbosa, Marcus Banks)",7.0
5,200610310LAL,LAL,LAL,2006-10-31,Andrew Bynum,Lamar Odom,Luke Walton,Sasha Vujacic,Smush Parker,Boris Diaw,Kurt Thomas,Raja Bell,Shawn Marion,Steve Nash,"(Andrew Bynum, Lamar Odom, Luke Walton, Sasha Vujacic, Smush Parker)","(Boris Diaw, Kurt Thomas, Raja Bell, Shawn Marion, Steve Nash)",6.0
2,200610310LAL,LAL,LAL,2006-10-31,Andrew Bynum,Lamar Odom,Luke Walton,Maurice Evans,Smush Parker,Boris Diaw,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,"(Andrew Bynum, Lamar Odom, Luke Walton, Maurice Evans, Smush Parker)","(Boris Diaw, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)",5.0
3,200610310LAL,LAL,LAL,2006-10-31,Andrew Bynum,Lamar Odom,Luke Walton,Maurice Evans,Smush Parker,Kurt Thomas,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,"(Andrew Bynum, Lamar Odom, Luke Walton, Maurice Evans, Smush Parker)","(Kurt Thomas, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)",4.0
1,200610310LAL,LAL,LAL,2006-10-31,Andrew Bynum,Lamar Odom,Luke Walton,Maurice Evans,Smush Parker,Boris Diaw,Kurt Thomas,Raja Bell,Shawn Marion,Steve Nash,"(Andrew Bynum, Lamar Odom, Luke Walton, Maurice Evans, Smush Parker)","(Boris Diaw, Kurt Thomas, Raja Bell, Shawn Marion, Steve Nash)",3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219771,201504150TOR,TOR,TOR,2015-04-15,DeMar DeRozan,Jonas Valanciunas,Kyle Lowry,Patrick Patterson,Terrence Ross,Gerald Henderson,Jason Maxiell,Kemba Walker,Marvin Williams,Troy Daniels,"(DeMar DeRozan, Jonas Valanciunas, Kyle Lowry, Patrick Patterson, Terrence Ross)","(Gerald Henderson, Jason Maxiell, Kemba Walker, Marvin Williams, Troy Daniels)",2.0
219764,201504150TOR,TOR,TOR,2015-04-15,Amir Johnson,DeMar DeRozan,James Johnson,Kyle Lowry,Patrick Patterson,Brian Roberts,Gerald Henderson,Jason Maxiell,Noah Vonleh,Troy Daniels,"(Amir Johnson, DeMar DeRozan, James Johnson, Kyle Lowry, Patrick Patterson)","(Brian Roberts, Gerald Henderson, Jason Maxiell, Noah Vonleh, Troy Daniels)",1.0
219765,201504150TOR,TOR,TOR,2015-04-15,Amir Johnson,Greivis Vasquez,James Johnson,Kyle Lowry,Patrick Patterson,Brian Roberts,Gerald Henderson,Jason Maxiell,Noah Vonleh,Troy Daniels,"(Amir Johnson, Greivis Vasquez, James Johnson, Kyle Lowry, Patrick Patterson)","(Brian Roberts, Gerald Henderson, Jason Maxiell, Noah Vonleh, Troy Daniels)",1.0
219767,201504150TOR,TOR,TOR,2015-04-15,Amir Johnson,Greivis Vasquez,Kyle Lowry,Patrick Patterson,Terrence Ross,Bismack Biyombo,Brian Roberts,Jeffery Taylor,Noah Vonleh,Troy Daniels,"(Amir Johnson, Greivis Vasquez, Kyle Lowry, Patrick Patterson, Terrence Ross)","(Bismack Biyombo, Brian Roberts, Jeffery Taylor, Noah Vonleh, Troy Daniels)",1.0


## 6. Additional Views and Analyses

In this section, we implement several new views to uncover hidden patterns:

1. **Frequency Analysis by Player Position:** How often does each player appear in a given home position (home_0, home_1, etc.)?
2. **Outcome Analysis by Home Lineup:** What are the win/loss outcomes (or average outcome) for different home lineups?
3. **Common Home Players Against Specific Away Lineups:** Which home players are most frequently used against a given away lineup?
4. **Lineup Variation Over Time:** How does the number of unique lineups change over time?

### 6.1 Frequency Analysis by Player Position

In [10]:
# Analyze frequency of players in each home position
for pos in ['home_0', 'home_1', 'home_2', 'home_3', 'home_4']:
    print(f"Frequency for {pos}:")
    print(df_filtered[pos].value_counts().head(10))
    print("---")

Frequency for home_0:
home_0
Andre Iguodala       5150
Al Jefferson         4492
Amar'e Stoudemire    3804
Boris Diaw           3502
Al Horford           3340
Al Harrington        3129
Beno Udrih           2951
Andrew Bogut         2652
Anderson Varejao     2650
Andray Blatche       2532
Name: count, dtype: int64
---
Frequency for home_1:
home_1
Dwyane Wade        2468
Carmelo Anthony    2446
David West         2433
Dirk Nowitzki      2258
Dwight Howard      2216
Andre Miller       2139
Deron Williams     2029
David Lee          1985
Kevin Durant       1983
Jamal Crawford     1982
Name: count, dtype: int64
---
Frequency for home_2:
home_2
LaMarcus Aldridge    2340
Josh Smith           2114
LeBron James         1864
J.R. Smith           1846
Kobe Bryant          1821
Jarrett Jack         1815
Deron Williams       1763
Paul Pierce          1761
Lamar Odom           1721
Mike Conley          1706
Name: count, dtype: int64
---
Frequency for home_3:
home_3
Tim Duncan       3281
Monta Ellis 

### 6.2 Outcome Analysis by Home Lineup

We group by the home lineup and calculate the average outcome and the number of games for each unique lineup. (Note: The `outcome` column typically indicates win/loss; you may need to adjust if using different metrics.)

In [11]:
# Ensure the dataset is sorted by game and starting_min
df_sorted = df_filtered.sort_values(['game', 'starting_min']).copy()

# Compute the duration of each lineup segment for each game.
# For each game, duration = next starting_min - current starting_min.
# For the last segment in each game, duration = 48 - current starting_min.
df_sorted['duration'] = df_sorted.groupby('game')['starting_min'].transform(lambda x: x.shift(-1) - x)
df_sorted['duration'] = df_sorted['duration'].fillna(48 - df_sorted['starting_min'])

# Group by season, home_team, and lineup columns to compute:
# - games: number of segments (or games) in which the lineup was used
# - avg_outcome: the average outcome for those segments
# - total_time: the sum of durations (i.e., total minutes played) for that lineup
lineup_outcomes = df_sorted.groupby(
    ['season', 'home_team', 'home_0', 'home_1', 'home_2', 'home_3', 'home_4', 'home_lineup']
).agg(
    games=('game', 'count'),
    avg_outcome=('outcome', 'mean'),
    total_time=('duration', 'sum')
).reset_index()

# Sort the results by season, then home_team, then by number of games (descending)
all_lineup_outcomes = lineup_outcomes.sort_values(['season', 'home_team', 'games'])

print(all_lineup_outcomes)


       season home_team           home_0           home_1           home_2  \
0        2007       ATL  Anthony Johnson  Esteban Batista       Josh Smith   
8        2007       ATL  Anthony Johnson      Joe Johnson   Josh Childress   
10       2007       ATL  Anthony Johnson      Joe Johnson  Marvin Williams   
15       2007       ATL  Anthony Johnson   Josh Childress       Josh Smith   
19       2007       ATL  Anthony Johnson   Josh Childress       Josh Smith   
...       ...       ...              ...              ...              ...   
59907    2015       WAS     Bradley Beal        John Wall   Kris Humphries   
59856    2015       WAS     Andre Miller   Kevin Seraphin   Kris Humphries   
59992    2015       WAS   Garrett Temple        John Wall    Marcin Gortat   
59908    2015       WAS     Bradley Beal        John Wall   Kris Humphries   
59918    2015       WAS     Bradley Beal        John Wall    Marcin Gortat   

                 home_3          home_4  \
0            Royal I

In [12]:
all_lineup_outcomes

,season,home_team,home_0,home_1,home_2,home_3,home_4,home_lineup,games,avg_outcome,total_time
0,2007,ATL,Anthony Johnson,Esteban Batista,Josh Smith,Royal Ivey,Solomon Jones,"(Anthony Johnson, Esteban Batista, Josh Smith, Royal Ivey, Solomon Jones)",1,-1.000000,1.0
8,2007,ATL,Anthony Johnson,Joe Johnson,Josh Childress,Marvin Williams,Tyronn Lue,"(Anthony Johnson, Joe Johnson, Josh Childress, Marvin Williams, Tyronn Lue)",1,-1.000000,4.0
10,2007,ATL,Anthony Johnson,Joe Johnson,Marvin Williams,Solomon Jones,Speedy Claxton,"(Anthony Johnson, Joe Johnson, Marvin Williams, Solomon Jones, Speedy Claxton)",1,-1.000000,3.0
15,2007,ATL,Anthony Johnson,Josh Childress,Josh Smith,Marvin Williams,Tyronn Lue,"(Anthony Johnson, Josh Childress, Josh Smith, Marvin Williams, Tyronn Lue)",1,1.000000,1.0
19,2007,ATL,Anthony Johnson,Josh Childress,Josh Smith,Shelden Williams,Solomon Jones,"(Anthony Johnson, Josh Childress, Josh Smith, Shelden Williams, Solomon Jones)",1,-1.000000,2.0
...,...,...,...,...,...,...,...,...,...,...,...
59907,2015,WAS,Bradley Beal,John Wall,Kris Humphries,Marcin Gortat,Otto Porter,"(Bradley Beal, John Wall, Kris Humphries, Marcin Gortat, Otto Porter)",24,-0.416667,50.0
59856,2015,WAS,Andre Miller,Kevin Seraphin,Kris Humphries,Otto Porter,Rasual Butler,"(Andre Miller, Kevin Seraphin, Kris Humphries, Otto Porter, Rasual Butler)",29,-0.034483,62.0
59992,2015,WAS,Garrett Temple,John Wall,Marcin Gortat,Nene Hilario,Paul Pierce,"(Garrett Temple, John Wall, Marcin Gortat, Nene Hilario, Paul Pierce)",40,-0.200000,132.0
59908,2015,WAS,Bradley Beal,John Wall,Kris Humphries,Marcin Gortat,Paul Pierce,"(Bradley Beal, John Wall, Kris Humphries, Marcin Gortat, Paul Pierce)",71,-0.014085,206.0


In [13]:
lineup_outcomes

,season,home_team,home_0,home_1,home_2,home_3,home_4,home_lineup,games,avg_outcome,total_time
0,2007,ATL,Anthony Johnson,Esteban Batista,Josh Smith,Royal Ivey,Solomon Jones,"(Anthony Johnson, Esteban Batista, Josh Smith, Royal Ivey, Solomon Jones)",1,-1.000000,1.0
1,2007,ATL,Anthony Johnson,Esteban Batista,Marvin Williams,Royal Ivey,Shelden Williams,"(Anthony Johnson, Esteban Batista, Marvin Williams, Royal Ivey, Shelden Williams)",2,0.000000,3.0
2,2007,ATL,Anthony Johnson,Joe Johnson,Josh Childress,Josh Smith,Marvin Williams,"(Anthony Johnson, Joe Johnson, Josh Childress, Josh Smith, Marvin Williams)",3,-1.000000,11.0
3,2007,ATL,Anthony Johnson,Joe Johnson,Josh Childress,Josh Smith,Solomon Jones,"(Anthony Johnson, Joe Johnson, Josh Childress, Josh Smith, Solomon Jones)",2,0.000000,3.0
4,2007,ATL,Anthony Johnson,Joe Johnson,Josh Childress,Josh Smith,Zaza Pachulia,"(Anthony Johnson, Joe Johnson, Josh Childress, Josh Smith, Zaza Pachulia)",6,-0.333333,9.0
...,...,...,...,...,...,...,...,...,...,...,...
60031,2015,WAS,Kevin Seraphin,Kris Humphries,Martell Webster,Ramon Sessions,Rasual Butler,"(Kevin Seraphin, Kris Humphries, Martell Webster, Ramon Sessions, Rasual Butler)",5,-0.600000,11.0
60032,2015,WAS,Kevin Seraphin,Kris Humphries,Otto Porter,Ramon Sessions,Rasual Butler,"(Kevin Seraphin, Kris Humphries, Otto Porter, Ramon Sessions, Rasual Butler)",1,-1.000000,2.0
60033,2015,WAS,Kevin Seraphin,Kris Humphries,Ramon Sessions,Rasual Butler,Will Bynum,"(Kevin Seraphin, Kris Humphries, Ramon Sessions, Rasual Butler, Will Bynum)",2,0.000000,3.0
60034,2015,WAS,Kevin Seraphin,Marcin Gortat,Martell Webster,Ramon Sessions,Rasual Butler,"(Kevin Seraphin, Marcin Gortat, Martell Webster, Ramon Sessions, Rasual Butler)",2,0.000000,2.0


### 6.3 Most Common Home Players Against Specific Away Lineups

For each unique away lineup, we analyze which home players appear most frequently. Here we melt the home player columns into one long format and then group by away lineup and player.

In [14]:
# Melt the home player columns to analyze individual player frequency per team per season
home_players = df_filtered.melt(
    id_vars=['season', 'game', 'home_team', 'away_team', 'away_lineup', 'date'],
    value_vars=['home_0', 'home_1', 'home_2', 'home_3', 'home_4'],
    var_name='home_position',
    value_name='home_player'
)

# Count the frequency of each home player per away lineup, per team, and per season
common_players_vs_away = home_players.groupby(['season', 'home_team', 'away_team', 'away_lineup', 'home_player']).size().reset_index(name='count')

# Sort by frequency of occurrence
common_players_vs_away = common_players_vs_away.sort_values(['season', 'count'], ascending=[True, False])


common_players_vs_away

,season,home_team,away_team,away_lineup,home_player,count
16320,2007,DAL,MIN,"(Kevin Garnett, Mark Blount, Mike James, Ricky Davis, Trenton Hassell)",Dirk Nowitzki,15
16325,2007,DAL,MIN,"(Kevin Garnett, Mark Blount, Mike James, Ricky Davis, Trenton Hassell)",Josh Howard,13
18583,2007,DEN,HOU,"(Dikembe Mutombo, Juwan Howard, Rafer Alston, Shane Battier, Tracy McGrady)",Allen Iverson,13
39556,2007,MEM,MIN,"(Kevin Garnett, Mark Blount, Mike James, Ricky Davis, Trenton Hassell)",Mike Miller,13
7866,2007,CHA,NYK,"(Eddy Curry, Malik Rose, Mardy Collins, Nate Robinson, Steve Francis)",Walter Herrmann,12
...,...,...,...,...,...,...
745790,2015,WAS,UTA,"(Enes Kanter, Gordon Hayward, Rodney Hood, Rudy Gobert, Trey Burke)",Paul Pierce,1
745793,2015,WAS,UTA,"(Gordon Hayward, Rodney Hood, Rudy Gobert, Trevor Booker, Trey Burke)",Kris Humphries,1
745795,2015,WAS,UTA,"(Gordon Hayward, Rodney Hood, Rudy Gobert, Trevor Booker, Trey Burke)",Nene Hilario,1
745796,2015,WAS,UTA,"(Gordon Hayward, Rodney Hood, Rudy Gobert, Trevor Booker, Trey Burke)",Paul Pierce,1


In [15]:
common_players_vs_away

,season,home_team,away_team,away_lineup,home_player,count
16320,2007,DAL,MIN,"(Kevin Garnett, Mark Blount, Mike James, Ricky Davis, Trenton Hassell)",Dirk Nowitzki,15
16325,2007,DAL,MIN,"(Kevin Garnett, Mark Blount, Mike James, Ricky Davis, Trenton Hassell)",Josh Howard,13
18583,2007,DEN,HOU,"(Dikembe Mutombo, Juwan Howard, Rafer Alston, Shane Battier, Tracy McGrady)",Allen Iverson,13
39556,2007,MEM,MIN,"(Kevin Garnett, Mark Blount, Mike James, Ricky Davis, Trenton Hassell)",Mike Miller,13
7866,2007,CHA,NYK,"(Eddy Curry, Malik Rose, Mardy Collins, Nate Robinson, Steve Francis)",Walter Herrmann,12
...,...,...,...,...,...,...
745790,2015,WAS,UTA,"(Enes Kanter, Gordon Hayward, Rodney Hood, Rudy Gobert, Trey Burke)",Paul Pierce,1
745793,2015,WAS,UTA,"(Gordon Hayward, Rodney Hood, Rudy Gobert, Trevor Booker, Trey Burke)",Kris Humphries,1
745795,2015,WAS,UTA,"(Gordon Hayward, Rodney Hood, Rudy Gobert, Trevor Booker, Trey Burke)",Nene Hilario,1
745796,2015,WAS,UTA,"(Gordon Hayward, Rodney Hood, Rudy Gobert, Trevor Booker, Trey Burke)",Paul Pierce,1


## 7. Conclusions and Next Steps

We have now organized the initial exploratory code and extended our analysis with additional views:

- **Frequency by Position:** Reveals which players appear most frequently in each home position.
- **Outcome Analysis:** Shows the performance (win/loss average) of different home lineups.
- **Common Matchups:** Identifies which home players are used most often against particular away lineups.
- **Temporal Trends:** Examines lineup variation over time.

These insights can help guide further feature engineering—such as adding a "lineup stability" score, network centrality measures for player synergy, or clustering of lineups—to improve the accuracy, precision, and recall of our missing-player prediction model.

Feel free to extend the analysis by exploring additional correlations (e.g., linking player positions if that information is available externally) or by applying association rule mining to detect frequent player combinations.

Let's continue discussing how we can further manipulate these data tables to gain even deeper insights.

In [16]:
lineup_outcomes.sort_values(['season', 'games'], ascending=[True, False])

,season,home_team,home_0,home_1,home_2,home_3,home_4,home_lineup,games,avg_outcome,total_time
3968,2007,MIN,Kevin Garnett,Mark Blount,Mike James,Ricky Davis,Trenton Hassell,"(Kevin Garnett, Mark Blount, Mike James, Ricky Davis, Trenton Hassell)",126,-0.301587,490.0
6684,2007,WAS,Antawn Jamison,Brendan Haywood,Caron Butler,DeShawn Stevenson,Gilbert Arenas,"(Antawn Jamison, Brendan Haywood, Caron Butler, DeShawn Stevenson, Gilbert Arenas)",108,-0.111111,352.0
6397,2007,UTA,Andrei Kirilenko,Carlos Boozer,Derek Fisher,Deron Williams,Mehmet Okur,"(Andrei Kirilenko, Carlos Boozer, Derek Fisher, Deron Williams, Mehmet Okur)",101,-0.049505,338.0
1521,2007,DAL,Devin Harris,Dirk Nowitzki,Erick Dampier,Jason Terry,Josh Howard,"(Devin Harris, Dirk Nowitzki, Erick Dampier, Jason Terry, Josh Howard)",96,0.145833,316.0
5218,2007,PHO,Amar'e Stoudemire,Boris Diaw,Raja Bell,Shawn Marion,Steve Nash,"(Amar'e Stoudemire, Boris Diaw, Raja Bell, Shawn Marion, Steve Nash)",94,-0.021277,355.0
...,...,...,...,...,...,...,...,...,...,...,...
60026,2015,WAS,John Wall,Marcin Gortat,Nene Hilario,Paul Pierce,Ramon Sessions,"(John Wall, Marcin Gortat, Nene Hilario, Paul Pierce, Ramon Sessions)",1,-1.000000,1.0
60028,2015,WAS,John Wall,Marcin Gortat,Nene Hilario,Paul Pierce,Will Bynum,"(John Wall, Marcin Gortat, Nene Hilario, Paul Pierce, Will Bynum)",1,-1.000000,3.0
60030,2015,WAS,John Wall,Martell Webster,Nene Hilario,Otto Porter,Paul Pierce,"(John Wall, Martell Webster, Nene Hilario, Otto Porter, Paul Pierce)",1,1.000000,1.0
60032,2015,WAS,Kevin Seraphin,Kris Humphries,Otto Porter,Ramon Sessions,Rasual Butler,"(Kevin Seraphin, Kris Humphries, Otto Porter, Ramon Sessions, Rasual Butler)",1,-1.000000,2.0


In [17]:
# Extract the starting lineup for each game (row with the smallest starting_min per game)
starting_lineups = df_filtered.sort_values('starting_min').groupby(['season', 'game']).first().reset_index()

# Group by season, home_team, and home_lineup to compute number of games and the average outcome
starting_lineup_stats = starting_lineups.groupby(['season', 'home_team', 'home_lineup']).agg(
    games_count=('game', 'count'),
    avg_outcome=('outcome', 'mean')
).reset_index()

# For each season and home_team, select the lineup with the maximum games_count (most frequently used lineup)
idx = starting_lineup_stats.groupby(['season', 'home_team'])['games_count'].idxmax()
most_common_starting_lineups = starting_lineup_stats.loc[idx].reset_index(drop=True)

# Sort results to see the most used lineups first
most_common_starting_lineups = most_common_starting_lineups.sort_values(['season', 'games_count'], ascending=[True, False])

# Display the result
print(most_common_starting_lineups)


     season home_team  \
16     2007       MIN   
5      2007       DAL   
20     2007       ORL   
22     2007       PHO   
3      2007       CHI   
..      ...       ...   
261    2015       ORL   
257    2015       MIN   
259    2015       NYK   
243    2015       CHA   
262    2015       PHI   

                                                                                     home_lineup  \
16                        (Kevin Garnett, Mark Blount, Mike James, Ricky Davis, Trenton Hassell)   
5                         (Devin Harris, Dirk Nowitzki, Erick Dampier, Jason Terry, Josh Howard)   
20                        (Dwight Howard, Grant Hill, Hedo Turkoglu, Jameer Nelson, Tony Battie)   
22                          (Amar'e Stoudemire, Boris Diaw, Raja Bell, Shawn Marion, Steve Nash)   
3                                 (Ben Gordon, Ben Wallace, Kirk Hinrich, Luol Deng, P.J. Brown)   
..                                                                                           ...   

In [16]:
most_common_starting_lineups.sort_values('season', ascending=True)

,season,home_team,home_lineup,games_count,avg_outcome
16,2007,MIN,"(Kevin Garnett, Mark Blount, Mike James, Ricky Davis, Trenton Hassell)",26,-0.076923
5,2007,DAL,"(Devin Harris, Dirk Nowitzki, Erick Dampier, Jason Terry, Josh Howard)",23,0.304348
20,2007,ORL,"(Dwight Howard, Grant Hill, Hedo Turkoglu, Jameer Nelson, Tony Battie)",23,0.217391
22,2007,PHO,"(Amar'e Stoudemire, Boris Diaw, Raja Bell, Shawn Marion, Steve Nash)",22,0.363636
3,2007,CHI,"(Ben Gordon, Ben Wallace, Kirk Hinrich, Luol Deng, P.J. Brown)",20,0.400000
...,...,...,...,...,...
264,2015,POR,"(Damian Lillard, LaMarcus Aldridge, Nicolas Batum, Robin Lopez, Wesley Matthews)",16,-0.250000
266,2015,SAS,"(Danny Green, Kawhi Leonard, Tiago Splitter, Tim Duncan, Tony Parker)",15,0.866667
265,2015,SAC,"(Ben McLemore, Darren Collison, DeMarcus Cousins, Jason Thompson, Rudy Gay)",15,0.200000
241,2015,BOS,"(Avery Bradley, Brandon Bass, Evan Turner, Marcus Smart, Tyler Zeller)",14,0.428571


In [18]:
# Group by season, home_team, and home_lineup to calculate the number of games and average outcome
team_lineup_success = df_filtered.groupby(['season', 'home_team', 'home_lineup']).agg(
    games=('game', 'count'),
    avg_outcome=('outcome', 'mean')
).reset_index()

# Optionally, filter out lineups with very few appearances to avoid outliers
min_games = 5  # Adjust threshold as needed
team_lineup_success = team_lineup_success[team_lineup_success['games'] >= min_games]

# For each season and home_team, select the lineup with the highest average outcome
idx = team_lineup_success.groupby(['season', 'home_team'])['avg_outcome'].idxmax()
most_successful_lineups = team_lineup_success.loc[idx].reset_index(drop=True)

# Sort results for easier reading (by season and success rate)
most_successful_lineups = most_successful_lineups.sort_values(['season', 'avg_outcome'], ascending=[True, False])

# Display the final DataFrame
print(most_successful_lineups)


     season home_team  \
12     2007       LAL   
24     2007       SAC   
3      2007       CHI   
27     2007       TOR   
1      2007       BOS   
..      ...       ...   
253    2015       LAL   
252    2015       LAC   
268    2015       UTA   
259    2015       NYK   
265    2015       SAC   

                                                                    home_lineup  \
12    (Kobe Bryant, Lamar Odom, Ronny Turiaf, Sasha Vujacic, Shammond Williams)   
24    (Brad Miller, Corliss Williamson, John Salmons, Kevin Martin, Mike Bibby)   
3           (Adrian Griffin, Ben Wallace, Chris Duhon, Kirk Hinrich, Luol Deng)   
27   (Chris Bosh, Joey Graham, Jorge Garbajosa, Jose Calderon, Morris Peterson)   
1       (Allan Ray, Gerald Green, Kevinn Pinkney, Leon Powe, Sebastian Telfair)   
..                                                                          ...   
253            (Ed Davis, Jeremy Lin, Jordan Hill, Kobe Bryant, Wesley Johnson)   
252          (Chris Paul, DeAndre J

In [19]:
pd.set_option('display.max_colwidth', None)
most_successful_lineups

,season,home_team,home_lineup,games,avg_outcome
12,2007,LAL,"(Kobe Bryant, Lamar Odom, Ronny Turiaf, Sasha Vujacic, Shammond Williams)",5,1.000000
24,2007,SAC,"(Brad Miller, Corliss Williamson, John Salmons, Kevin Martin, Mike Bibby)",5,1.000000
3,2007,CHI,"(Adrian Griffin, Ben Wallace, Chris Duhon, Kirk Hinrich, Luol Deng)",7,0.714286
27,2007,TOR,"(Chris Bosh, Joey Graham, Jorge Garbajosa, Jose Calderon, Morris Peterson)",7,0.714286
1,2007,BOS,"(Allan Ray, Gerald Green, Kevinn Pinkney, Leon Powe, Sebastian Telfair)",6,0.666667
...,...,...,...,...,...
253,2015,LAL,"(Ed Davis, Jeremy Lin, Jordan Hill, Kobe Bryant, Wesley Johnson)",12,0.500000
252,2015,LAC,"(Chris Paul, DeAndre Jordan, Glen Davis, J.J. Redick, Matt Barnes)",7,0.428571
268,2015,UTA,"(Dante Exum, Elijah Millsap, Rudy Gobert, Trevor Booker, Trey Burke)",10,0.400000
259,2015,NYK,"(Alexey Shved, Cole Aldrich, Jason Smith, Shane Larkin, Travis Wear)",6,0.333333


In [17]:
# Extract home players, including season information, and rename the team column
home_players = df_filtered.melt(
    id_vars=['game', 'season', 'home_team'],
    value_vars=['home_0', 'home_1', 'home_2', 'home_3', 'home_4'],
    var_name='position',
    value_name='player'
).rename(columns={'home_team': 'team'})

# Extract away players, including season information, and rename the team column
away_players = df_filtered.melt(
    id_vars=['game', 'season', 'away_team'],
    value_vars=['away_0', 'away_1', 'away_2', 'away_3', 'away_4'],
    var_name='position',
    value_name='player'
).rename(columns={'away_team': 'team'})

# Combine home and away players into one DataFrame
all_players = pd.concat(
    [home_players[['game', 'season', 'team', 'player']], 
     away_players[['game', 'season', 'team', 'player']]],
    ignore_index=True
)

# Group by team and season to get unique players for each team in each season
team_players = all_players.groupby(['team', 'season'])['player'].unique().reset_index()

# Optionally, convert the array of players to a sorted, comma-separated string for easier reading
team_players['players'] = team_players['player'].apply(lambda x: ', '.join(sorted(x)))
team_players = team_players[['team', 'season', 'players']]

team_players


,team,season,players
0,ATL,2007,"Anthony Johnson, Cedric Bozeman, Dijon Thompson, Esteban Batista, Jeremy Richardson, Joe Johnson, Josh Childress, Josh Smith, Lorenzen Wright, Marvin Williams, Matt Freije, Royal Ivey, Salim Stoudamire, Shelden Williams, Solomon Jones, Speedy Claxton, Stanislav Medvedenko, Tyronn Lue, Zaza Pachulia"
1,ATL,2008,"Acie Law, Al Horford, Anthony Johnson, Jeremy Richardson, Joe Johnson, Josh Childress, Josh Smith, Lorenzen Wright, Mario West, Marvin Williams, Mike Bibby, Salim Stoudamire, Shelden Williams, Solomon Jones, Tyronn Lue, Zaza Pachulia"
2,ATL,2009,"Acie Law, Al Horford, Joe Johnson, Josh Smith, Mario West, Marvin Williams, Maurice Evans, Mike Bibby, Othello Hunter, Randolph Morris, Ronald Murray, Solomon Jones, Speedy Claxton, Thomas Gardner, Zaza Pachulia"
3,ATL,2010,"Al Horford, Jamal Crawford, Jason Collins, Jeff Teague, Joe Johnson, Joe Smith, Josh Smith, Mario West, Marvin Williams, Maurice Evans, Mike Bibby, Othello Hunter, Randolph Morris, Zaza Pachulia"
4,ATL,2011,"Al Horford, Damien Wilkins, Etan Thomas, Hilton Armstrong, Jamal Crawford, Jason Collins, Jeff Teague, Joe Johnson, Jordan Crawford, Josh Powell, Josh Smith, Kirk Hinrich, Marvin Williams, Maurice Evans, Mike Bibby, Pape Sy, Zaza Pachulia"
...,...,...,...
265,WAS,2011,"Al Thornton, Alonzo Gee, Andray Blatche, Cartier Martin, Gilbert Arenas, Hamady N'Diaye, Hilton Armstrong, JaVale McGee, John Wall, Jordan Crawford, Josh Howard, Kevin Seraphin, Kirk Hinrich, Larry Owens, Lester Hudson, Maurice Evans, Mike Bibby, Mustafa Shakur, Nick Young, Othyus Jeffers, Rashard Lewis, Trevor Booker, Yi Jianlian"
266,WAS,2012,"Andray Blatche, Brian Cook, Cartier Martin, Chris Singleton, Edwin Ubiles, JaVale McGee, James Singleton, Jan Vesely, John Wall, Jordan Crawford, Kevin Seraphin, Maurice Evans, Morris Almond, Nene Hilario, Nick Young, Rashard Lewis, Roger Mason, Ronny Turiaf, Shelvin Mack, Trevor Booker"
267,WAS,2013,"A.J. Price, Bradley Beal, Cartier Martin, Chris Singleton, Earl Barron, Emeka Okafor, Garrett Temple, Jan Vesely, Jannero Pargo, Jason Collins, John Wall, Jordan Crawford, Kevin Seraphin, Martell Webster, Nene Hilario, Shaun Livingston, Shelvin Mack, Trevor Ariza, Trevor Booker"
268,WAS,2014,"Al Harrington, Andre Miller, Bradley Beal, Chris Singleton, Drew Gooden, Eric Maynor, Garrett Temple, Glen Rice, Jan Vesely, John Wall, Kevin Seraphin, Marcin Gortat, Martell Webster, Nene Hilario, Otto Porter, Trevor Ariza, Trevor Booker"


In [21]:
# Melt the home player columns to analyze player participation
home_players = df_filtered.melt(
    id_vars=['game', 'season', 'home_team'],
    value_vars=['home_0', 'home_1', 'home_2', 'home_3', 'home_4'],
    var_name='home_position',
    value_name='player'
).rename(columns={'home_team': 'team'})

# Melt the away player columns
away_players = df_filtered.melt(
    id_vars=['game', 'season', 'away_team'],
    value_vars=['away_0', 'away_1', 'away_2', 'away_3', 'away_4'],
    var_name='away_position',
    value_name='player'
).rename(columns={'away_team': 'team'})

# Combine both home and away players into one DataFrame
all_players = pd.concat(
    [home_players[['game', 'season', 'team', 'player']], 
     away_players[['game', 'season', 'team', 'player']]],
    ignore_index=True
)

# Count how many games each player appeared in for each team in each season
player_game_counts = all_players.groupby(['season', 'team', 'player'])['game'].nunique().reset_index()

# Rename the column for clarity
player_game_counts.rename(columns={'game': 'games_played'}, inplace=True)

# Sort the results for better readability
player_game_counts = player_game_counts.sort_values(['season', 'team', 'games_played'], ascending=[True, True, False])


player_game_counts

,season,team,player,games_played
13,2007,ATL,Shelden Williams,81
7,2007,ATL,Josh Smith,72
18,2007,ATL,Zaza Pachulia,72
8,2007,ATL,Lorenzen Wright,67
9,2007,ATL,Marvin Williams,64
...,...,...,...,...
4725,2015,WAS,Ramon Sessions,28
4713,2015,WAS,DeJuan Blair,26
4728,2015,WAS,Will Bynum,6
4716,2015,WAS,Glen Rice,5


In [22]:
# First, aggregate unique players by team and season without converting to string
team_players_df = all_players.groupby(['team', 'season'])['player'].unique().reset_index()

# Convert the 'player' column (which is an array) to a list for each row
team_players_df['player'] = team_players_df['player'].apply(list)

# Now, build a nested dictionary: {team: {season: [players]}}
rosters_dict = {}
for _, row in team_players_df.iterrows():
    team = row['team']
    season = row['season']
    players_list = row['player']
    if team not in rosters_dict:
        rosters_dict[team] = {}
    rosters_dict[team][season] = sorted(players_list)  # sorted for consistency

# Print the dictionary
print(rosters_dict)


{'ATL': {2007: ['Anthony Johnson', 'Cedric Bozeman', 'Dijon Thompson', 'Esteban Batista', 'Jeremy Richardson', 'Joe Johnson', 'Josh Childress', 'Josh Smith', 'Lorenzen Wright', 'Marvin Williams', 'Matt Freije', 'Royal Ivey', 'Salim Stoudamire', 'Shelden Williams', 'Solomon Jones', 'Speedy Claxton', 'Stanislav Medvedenko', 'Tyronn Lue', 'Zaza Pachulia'], 2008: ['Acie Law', 'Al Horford', 'Anthony Johnson', 'Jeremy Richardson', 'Joe Johnson', 'Josh Childress', 'Josh Smith', 'Lorenzen Wright', 'Mario West', 'Marvin Williams', 'Mike Bibby', 'Salim Stoudamire', 'Shelden Williams', 'Solomon Jones', 'Tyronn Lue', 'Zaza Pachulia'], 2009: ['Acie Law', 'Al Horford', 'Joe Johnson', 'Josh Smith', 'Mario West', 'Marvin Williams', 'Maurice Evans', 'Mike Bibby', 'Othello Hunter', 'Randolph Morris', 'Ronald Murray', 'Solomon Jones', 'Speedy Claxton', 'Thomas Gardner', 'Zaza Pachulia'], 2010: ['Al Horford', 'Jamal Crawford', 'Jason Collins', 'Jeff Teague', 'Joe Johnson', 'Joe Smith', 'Josh Smith', 'Mari

In [18]:
# --- Compute Minutes Played by Each Player for Both Home and Away ---

# Sort the filtered DataFrame by game and starting_min, then compute the duration for each lineup segment.
df_sorted = df_filtered.sort_values(['game', 'starting_min']).copy()
# Duration is the difference between the current starting_min and the next one in the same game;
# for the last segment in each game, use (48 - current starting_min)
df_sorted['duration'] = df_sorted.groupby('game')['starting_min'].transform(lambda x: x.shift(-1) - x)
df_sorted['duration'] = df_sorted['duration'].fillna(48 - df_sorted['starting_min'])

# --- Compute minutes for home players ---
# Melt the home player columns to long format.
home_players = df_sorted.melt(
    id_vars=['game', 'season', 'home_team', 'duration'],
    value_vars=['home_0', 'home_1', 'home_2', 'home_3', 'home_4'],
    var_name='position',
    value_name='player'
)
# Rename the team column for consistency.
home_players = home_players.rename(columns={'home_team': 'team'})

# --- Compute minutes for away players ---
# Melt the away player columns similarly.
away_players = df_sorted.melt(
    id_vars=['game', 'season', 'away_team', 'duration'],
    value_vars=['away_0', 'away_1', 'away_2', 'away_3', 'away_4'],
    var_name='position',
    value_name='player'
)
away_players = away_players.rename(columns={'away_team': 'team'})

# Combine the two DataFrames
all_players_minutes = pd.concat([home_players[['game', 'season', 'team', 'player', 'duration']],
                                 away_players[['game', 'season', 'team', 'player', 'duration']]],
                                ignore_index=True)

# Group by team, season, and player to get total minutes played.
player_minutes = (
    all_players_minutes.groupby(['team', 'season', 'player'])['duration']
    .sum()
    .reset_index()
    .rename(columns={'duration': 'total_minutes'})
)

# Optional: sort for readability
player_minutes = player_minutes.sort_values(['team', 'season', 'total_minutes'], ascending=[True, True, False])
display(player_minutes)

# --- Build a Roster Dictionary Including Minutes Played ---
# This dictionary will have the structure:
# { team: { season: { player: total_minutes, ... }, ... }, ... }

rosters_with_minutes = {}
for _, row in player_minutes.iterrows():
    team = row['team']
    season = row['season']
    player = row['player']
    minutes = row['total_minutes']
    if team not in rosters_with_minutes:
        rosters_with_minutes[team] = {}
    if season not in rosters_with_minutes[team]:
        rosters_with_minutes[team][season] = {}
    rosters_with_minutes[team][season][player] = minutes

# Print the roster dictionary with minutes played
import pprint
pprint.pprint(rosters_with_minutes)


,team,season,player,total_minutes
7,ATL,2007,Josh Smith,2637.0
5,ATL,2007,Joe Johnson,2355.0
9,ATL,2007,Marvin Williams,2181.0
18,ATL,2007,Zaza Pachulia,2019.0
6,ATL,2007,Josh Childress,1982.0
...,...,...,...,...
4721,WAS,2015,Martell Webster,330.0
4713,WAS,2015,DeJuan Blair,153.0
4728,WAS,2015,Will Bynum,57.0
4716,WAS,2015,Glen Rice,42.0


{'ATL': {2007: {'Anthony Johnson': 732.0,
                'Cedric Bozeman': 197.0,
                'Dijon Thompson': 46.0,
                'Esteban Batista': 76.0,
                'Jeremy Richardson': 17.0,
                'Joe Johnson': 2355.0,
                'Josh Childress': 1982.0,
                'Josh Smith': 2637.0,
                'Lorenzen Wright': 1066.0,
                'Marvin Williams': 2181.0,
                'Matt Freije': 138.0,
                'Royal Ivey': 510.0,
                'Salim Stoudamire': 978.0,
                'Shelden Williams': 1502.0,
                'Solomon Jones': 642.0,
                'Speedy Claxton': 1061.0,
                'Stanislav Medvedenko': 72.0,
                'Tyronn Lue': 1469.0,
                'Zaza Pachulia': 2019.0},
         2008: {'Acie Law': 849.0,
                'Al Horford': 2550.0,
                'Anthony Johnson': 1127.0,
                'Jeremy Richardson': 73.0,
                'Joe Johnson': 3338.0,
                'Jos

In [24]:
rosters_with_minutes['TOR'][2015] 

{'Kyle Lowry': 2412.0,
 'Jonas Valanciunas': 2154.0,
 'DeMar DeRozan': 2117.0,
 'Terrence Ross': 2087.0,
 'Patrick Patterson': 2073.0,
 'Amir Johnson': 1983.0,
 'Greivis Vasquez': 1958.0,
 'Lou Williams': 1949.0,
 'James Johnson': 1356.0,
 'Tyler Hansbrough': 1047.0,
 'Chuck Hayes': 254.0,
 'Landry Fields': 206.0,
 'Greg Stiemsma': 50.0,
 'Lucas Nogueira': 18.0,
 'Bruno Caboclo': 16.0}

In [19]:
# --- Calculate minutes played by each player on each team for each season ---

# Sort the DataFrame by game and starting_min, then compute the duration for each lineup segment.
df_sorted = df_filtered.sort_values(['game', 'starting_min']).copy()
df_sorted['duration'] = df_sorted.groupby('game')['starting_min'].transform(lambda x: x.shift(-1) - x)
df_sorted['duration'] = df_sorted['duration'].fillna(48 - df_sorted['starting_min'])

# Melt the home player columns so each row corresponds to a player's appearance in a lineup segment.
# This includes the game, season, home_team, and duration (minutes played in that segment).
home_players = df_sorted.melt(
    id_vars=['game', 'season', 'home_team', 'duration'],
    value_vars=['home_0', 'home_1', 'home_2', 'home_3', 'home_4'],
    var_name='position',
    value_name='player'
)

# Group by team, season, and player to calculate the total minutes played.
player_minutes = home_players.groupby(['home_team', 'season', 'player'])['duration'].sum().reset_index()
player_minutes.rename(columns={'duration': 'total_minutes'}, inplace=True)

# Sort the results for easier reading.
player_minutes = player_minutes.sort_values(['home_team', 'season', 'total_minutes'], ascending=[True, True, False])

# Display the complete DataFrame
display(player_minutes)


,home_team,season,player,total_minutes
7,ATL,2007,Josh Smith,1322.0
5,ATL,2007,Joe Johnson,1120.0
9,ATL,2007,Marvin Williams,1107.0
6,ATL,2007,Josh Childress,1077.0
18,ATL,2007,Zaza Pachulia,1034.0
...,...,...,...,...
4668,WAS,2015,Martell Webster,141.0
4660,WAS,2015,DeJuan Blair,66.0
4663,WAS,2015,Glen Rice,8.0
4675,WAS,2015,Will Bynum,6.0


In [26]:
team_players

,team,season,players
0,ATL,2007,"Anthony Johnson, Cedric Bozeman, Dijon Thompson, Esteban Batista, Jeremy Richardson, Joe Johnson, Josh Childress, Josh Smith, Lorenzen Wright, Marvin Williams, Matt Freije, Royal Ivey, Salim Stoudamire, Shelden Williams, Solomon Jones, Speedy Claxton, Stanislav Medvedenko, Tyronn Lue, Zaza Pachulia"
1,ATL,2008,"Acie Law, Al Horford, Anthony Johnson, Jeremy Richardson, Joe Johnson, Josh Childress, Josh Smith, Lorenzen Wright, Mario West, Marvin Williams, Mike Bibby, Salim Stoudamire, Shelden Williams, Solomon Jones, Tyronn Lue, Zaza Pachulia"
2,ATL,2009,"Acie Law, Al Horford, Joe Johnson, Josh Smith, Mario West, Marvin Williams, Maurice Evans, Mike Bibby, Othello Hunter, Randolph Morris, Ronald Murray, Solomon Jones, Speedy Claxton, Thomas Gardner, Zaza Pachulia"
3,ATL,2010,"Al Horford, Jamal Crawford, Jason Collins, Jeff Teague, Joe Johnson, Joe Smith, Josh Smith, Mario West, Marvin Williams, Maurice Evans, Mike Bibby, Othello Hunter, Randolph Morris, Zaza Pachulia"
4,ATL,2011,"Al Horford, Damien Wilkins, Etan Thomas, Hilton Armstrong, Jamal Crawford, Jason Collins, Jeff Teague, Joe Johnson, Jordan Crawford, Josh Powell, Josh Smith, Kirk Hinrich, Marvin Williams, Maurice Evans, Mike Bibby, Pape Sy, Zaza Pachulia"
...,...,...,...
265,WAS,2011,"Al Thornton, Alonzo Gee, Andray Blatche, Cartier Martin, Gilbert Arenas, Hamady N'Diaye, Hilton Armstrong, JaVale McGee, John Wall, Jordan Crawford, Josh Howard, Kevin Seraphin, Kirk Hinrich, Larry Owens, Lester Hudson, Maurice Evans, Mike Bibby, Mustafa Shakur, Nick Young, Othyus Jeffers, Rashard Lewis, Trevor Booker, Yi Jianlian"
266,WAS,2012,"Andray Blatche, Brian Cook, Cartier Martin, Chris Singleton, Edwin Ubiles, JaVale McGee, James Singleton, Jan Vesely, John Wall, Jordan Crawford, Kevin Seraphin, Maurice Evans, Morris Almond, Nene Hilario, Nick Young, Rashard Lewis, Roger Mason, Ronny Turiaf, Shelvin Mack, Trevor Booker"
267,WAS,2013,"A.J. Price, Bradley Beal, Cartier Martin, Chris Singleton, Earl Barron, Emeka Okafor, Garrett Temple, Jan Vesely, Jannero Pargo, Jason Collins, John Wall, Jordan Crawford, Kevin Seraphin, Martell Webster, Nene Hilario, Shaun Livingston, Shelvin Mack, Trevor Ariza, Trevor Booker"
268,WAS,2014,"Al Harrington, Andre Miller, Bradley Beal, Chris Singleton, Drew Gooden, Eric Maynor, Garrett Temple, Glen Rice, Jan Vesely, John Wall, Kevin Seraphin, Marcin Gortat, Martell Webster, Nene Hilario, Otto Porter, Trevor Ariza, Trevor Booker"


In [28]:
most_common_starting_lineups

,season,home_team,home_lineup,games_count,avg_outcome
16,2007,MIN,"(Kevin Garnett, Mark Blount, Mike James, Ricky Davis, Trenton Hassell)",26,-0.076923
5,2007,DAL,"(Devin Harris, Dirk Nowitzki, Erick Dampier, Jason Terry, Josh Howard)",23,0.304348
20,2007,ORL,"(Dwight Howard, Grant Hill, Hedo Turkoglu, Jameer Nelson, Tony Battie)",23,0.217391
22,2007,PHO,"(Amar'e Stoudemire, Boris Diaw, Raja Bell, Shawn Marion, Steve Nash)",22,0.363636
3,2007,CHI,"(Ben Gordon, Ben Wallace, Kirk Hinrich, Luol Deng, P.J. Brown)",20,0.400000
...,...,...,...,...,...
261,2015,ORL,"(Dewayne Dedmon, Elfrid Payton, Nikola Vucevic, Tobias Harris, Victor Oladipo)",7,-0.142857
257,2015,MIN,"(Andrew Wiggins, Corey Brewer, Gorgui Dieng, Thaddeus Young, Zach LaVine)",6,-0.666667
259,2015,NYK,"(Andrea Bargnani, Lance Thomas, Langston Galloway, Lou Amundson, Shane Larkin)",6,-0.666667
244,2015,CHO,"(Al Jefferson, Cody Zeller, Gerald Henderson, Kemba Walker, Lance Stephenson)",5,-0.600000


In [27]:
most_successful_lineups

,season,home_team,home_lineup,games,avg_outcome
12,2007,LAL,"(Kobe Bryant, Lamar Odom, Ronny Turiaf, Sasha Vujacic, Shammond Williams)",5,1.000000
24,2007,SAC,"(Brad Miller, Corliss Williamson, John Salmons, Kevin Martin, Mike Bibby)",5,1.000000
3,2007,CHI,"(Adrian Griffin, Ben Wallace, Chris Duhon, Kirk Hinrich, Luol Deng)",7,0.714286
27,2007,TOR,"(Chris Bosh, Joey Graham, Jorge Garbajosa, Jose Calderon, Morris Peterson)",7,0.714286
1,2007,BOS,"(Allan Ray, Gerald Green, Kevinn Pinkney, Leon Powe, Sebastian Telfair)",6,0.666667
...,...,...,...,...,...
253,2015,LAL,"(Ed Davis, Jeremy Lin, Jordan Hill, Kobe Bryant, Wesley Johnson)",12,0.500000
252,2015,LAC,"(Chris Paul, DeAndre Jordan, Glen Davis, J.J. Redick, Matt Barnes)",7,0.428571
268,2015,UTA,"(Dante Exum, Elijah Millsap, Rudy Gobert, Trevor Booker, Trey Burke)",10,0.400000
259,2015,NYK,"(Alexey Shved, Cole Aldrich, Jason Smith, Shane Larkin, Travis Wear)",6,0.333333


In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# =============================================================================
# STEP 1: Compute Duration for Each Lineup Segment
# =============================================================================
df_sorted = df_filtered.sort_values(['game', 'starting_min']).copy()
df_sorted['duration'] = df_sorted.groupby('game')['starting_min'].transform(lambda x: x.shift(-1) - x)
df_sorted['duration'] = df_sorted['duration'].fillna(48 - df_sorted['starting_min'])

# =============================================================================
# STEP 2: Mean Encode Home Players
# =============================================================================
home_players = df_sorted.melt(
    id_vars=['game', 'season', 'home_team', 'away_lineup', 'duration', 'outcome'],
    value_vars=['home_0', 'home_1', 'home_2', 'home_3', 'home_4'],
    var_name='position',
    value_name='player'
).rename(columns={'home_team': 'team'})

home_players['weighted_outcome'] = home_players['outcome'] * home_players['duration']

player_outcome_home = home_players.groupby(['season', 'team', 'player']).agg(
    total_weighted_outcome=('weighted_outcome', 'sum'),
    total_minutes=('duration', 'sum')
).reset_index()
player_outcome_home['base_mean_encoding'] = player_outcome_home['total_weighted_outcome'] / player_outcome_home['total_minutes']

opp_occurrence_home = home_players.groupby(['season', 'team', 'player']).size().reset_index(name='total_opp_occurrences')
opp_lineup_counts_home = home_players.groupby(['season', 'team', 'player'])['away_lineup'].nunique().reset_index(name='distinct_away_lineups')
opp_stats_home = opp_occurrence_home.merge(opp_lineup_counts_home, on=['season', 'team', 'player'])
opp_stats_home['avg_opp_occurrence'] = opp_stats_home['total_opp_occurrences'] / opp_stats_home['distinct_away_lineups']

player_encoding_home = player_outcome_home.merge(
    opp_stats_home[['season', 'team', 'player', 'avg_opp_occurrence']],
    on=['season', 'team', 'player'], how='left'
)
player_encoding_home['avg_opp_occurrence'] = player_encoding_home['avg_opp_occurrence'].fillna(0)

lambda_param = 0.05
player_encoding_home['final_mean_encoding'] = player_encoding_home['base_mean_encoding'] * (1 + lambda_param * player_encoding_home['avg_opp_occurrence'])

# =============================================================================
# STEP 2b: Compute Position Occurrence Factor for Home Players
# =============================================================================
# Count how often each player appears in each home position.
home_position_counts = home_players.groupby(['season', 'team', 'player', 'position']).size().reset_index(name='position_count')
# Count total appearances per player (regardless of position)
total_appearances = home_players.groupby(['season', 'team', 'player']).size().reset_index(name='total_appearances')
# Merge and compute frequency for each position.
home_pos_stats = home_position_counts.merge(total_appearances, on=['season', 'team', 'player'])
home_pos_stats['position_freq'] = home_pos_stats['position_count'] / home_pos_stats['total_appearances']
# Build a dictionary for home position frequency keyed by (season, team, player, position)
pos_freq_dict_home = dict(zip(
    zip(home_pos_stats['season'], home_pos_stats['team'], home_pos_stats['player'], home_pos_stats['position']),
    home_pos_stats['position_freq']
))

# =============================================================================
# STEP 3: Mean Encode Away Players
# =============================================================================
away_players = df_sorted.melt(
    id_vars=['game', 'season', 'away_team', 'duration', 'outcome'],
    value_vars=['away_0', 'away_1', 'away_2', 'away_3', 'away_4'],
    var_name='position',
    value_name='player'
).rename(columns={'away_team': 'team'})

# Invert outcome for away players (since outcome is from home perspective)
away_players['away_outcome'] = 1 - away_players['outcome']  # if outcome is 1 for win, 0 for loss
away_players['weighted_outcome'] = away_players['away_outcome'] * away_players['duration']

player_outcome_away = away_players.groupby(['season', 'team', 'player']).agg(
    total_weighted_outcome=('weighted_outcome', 'sum'),
    total_minutes=('duration', 'sum')
).reset_index()
player_outcome_away['base_mean_encoding'] = player_outcome_away['total_weighted_outcome'] / player_outcome_away['total_minutes']

home_lineup_info = df_sorted[['game', 'home_lineup']].drop_duplicates()
away_players = away_players.merge(home_lineup_info, on='game', how='left')

opp_occurrence_away = away_players.groupby(['season', 'team', 'player']).size().reset_index(name='total_opp_occurrences')
opp_lineup_counts_away = away_players.groupby(['season', 'team', 'player'])['home_lineup'].nunique().reset_index(name='distinct_home_lineups')
opp_stats_away = opp_occurrence_away.merge(opp_lineup_counts_away, on=['season', 'team', 'player'])
opp_stats_away['avg_opp_occurrence'] = opp_stats_away['total_opp_occurrences'] / opp_stats_away['distinct_home_lineups']

player_encoding_away = player_outcome_away.merge(
    opp_stats_away[['season', 'team', 'player', 'avg_opp_occurrence']],
    on=['season', 'team', 'player'], how='left'
)
player_encoding_away['avg_opp_occurrence'] = player_encoding_away['avg_opp_occurrence'].fillna(0)
player_encoding_away['final_mean_encoding'] = player_encoding_away['base_mean_encoding'] * (1 + lambda_param * player_encoding_away['avg_opp_occurrence'])

# =============================================================================
# STEP 3b: Compute Position Occurrence Factor for Away Players
# =============================================================================
away_position_counts = away_players.groupby(['season', 'team', 'player', 'position']).size().reset_index(name='position_count')
total_appearances_away = away_players.groupby(['season', 'team', 'player']).size().reset_index(name='total_appearances')
away_pos_stats = away_position_counts.merge(total_appearances_away, on=['season', 'team', 'player'])
away_pos_stats['position_freq'] = away_pos_stats['position_count'] / away_pos_stats['total_appearances']
pos_freq_dict_away = dict(zip(
    zip(away_pos_stats['season'], away_pos_stats['team'], away_pos_stats['player'], away_pos_stats['position']),
    away_pos_stats['position_freq']
))



In [20]:
home_pos_stats

,season,team,player,position,position_count,total_appearances,position_freq
0,2007,ATL,Anthony Johnson,home_0,198,198,1.000000
1,2007,ATL,Cedric Bozeman,home_0,43,43,1.000000
2,2007,ATL,Dijon Thompson,home_0,13,13,1.000000
3,2007,ATL,Esteban Batista,home_0,3,6,0.500000
4,2007,ATL,Esteban Batista,home_1,3,6,0.500000
...,...,...,...,...,...,...,...
13839,2015,WAS,Ramon Sessions,home_4,64,117,0.547009
13840,2015,WAS,Rasual Butler,home_3,4,410,0.009756
13841,2015,WAS,Rasual Butler,home_4,406,410,0.990244
13842,2015,WAS,Toure' Murry,home_4,2,2,1.000000


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# =============================================================================
# STEP 1: Compute Duration for Each Lineup Segment
# =============================================================================
df_sorted = df_filtered.sort_values(['game', 'starting_min']).copy()
df_sorted['duration'] = df_sorted.groupby('game')['starting_min'].transform(lambda x: x.shift(-1) - x)
df_sorted['duration'] = df_sorted['duration'].fillna(48 - df_sorted['starting_min'])

# =============================================================================
# STEP 2: Mean Encode Home Players
# =============================================================================
home_players = df_sorted.melt(
    id_vars=['game', 'season', 'home_team', 'away_lineup', 'duration', 'outcome'],
    value_vars=['home_0', 'home_1', 'home_2', 'home_3', 'home_4'],
    var_name='position',
    value_name='player'
).rename(columns={'home_team': 'team'})

home_players['weighted_outcome'] = home_players['outcome'] * home_players['duration']

player_outcome_home = home_players.groupby(['season', 'team', 'player']).agg(
    total_weighted_outcome=('weighted_outcome', 'sum'),
    total_minutes=('duration', 'sum')
).reset_index()
player_outcome_home['base_mean_encoding'] = player_outcome_home['total_weighted_outcome'] / player_outcome_home['total_minutes']

opp_occurrence_home = home_players.groupby(['season', 'team', 'player']).size().reset_index(name='total_opp_occurrences')
opp_lineup_counts_home = home_players.groupby(['season', 'team', 'player'])['away_lineup'].nunique().reset_index(name='distinct_away_lineups')
opp_stats_home = opp_occurrence_home.merge(opp_lineup_counts_home, on=['season', 'team', 'player'])
opp_stats_home['avg_opp_occurrence'] = opp_stats_home['total_opp_occurrences'] / opp_stats_home['distinct_away_lineups']

player_encoding_home = player_outcome_home.merge(
    opp_stats_home[['season', 'team', 'player', 'avg_opp_occurrence']],
    on=['season', 'team', 'player'], how='left'
)
player_encoding_home['avg_opp_occurrence'] = player_encoding_home['avg_opp_occurrence'].fillna(0)

lambda_param = 0.05
player_encoding_home['final_mean_encoding'] = player_encoding_home['base_mean_encoding'] * (1 + lambda_param * player_encoding_home['avg_opp_occurrence'])

mean_encoding_dict_home = dict(zip(
    zip(player_encoding_home['season'], player_encoding_home['team'], player_encoding_home['player']),
    player_encoding_home['final_mean_encoding']
))

# =============================================================================
# STEP 3: Mean Encode Away Players
# =============================================================================
away_players = df_sorted.melt(
    id_vars=['game', 'season', 'away_team', 'duration', 'outcome'],
    value_vars=['away_0', 'away_1', 'away_2', 'away_3', 'away_4'],
    var_name='position',
    value_name='player'
).rename(columns={'away_team': 'team'})

# Invert outcome for away players (because outcome is from the home perspective)
away_players['away_outcome'] = -away_players['outcome']
away_players['weighted_outcome'] = away_players['away_outcome'] * away_players['duration']

player_outcome_away = away_players.groupby(['season', 'team', 'player']).agg(
    total_weighted_outcome=('weighted_outcome', 'sum'),
    total_minutes=('duration', 'sum')
).reset_index()
player_outcome_away['base_mean_encoding'] = player_outcome_away['total_weighted_outcome'] / player_outcome_away['total_minutes']

home_lineup_info = df_sorted[['game', 'home_lineup']].drop_duplicates()
away_players = away_players.merge(home_lineup_info, on='game', how='left')

opp_occurrence_away = away_players.groupby(['season', 'team', 'player']).size().reset_index(name='total_opp_occurrences')
opp_lineup_counts_away = away_players.groupby(['season', 'team', 'player'])['home_lineup'].nunique().reset_index(name='distinct_home_lineups')
opp_stats_away = opp_occurrence_away.merge(opp_lineup_counts_away, on=['season', 'team', 'player'])
opp_stats_away['avg_opp_occurrence'] = opp_stats_away['total_opp_occurrences'] / opp_stats_away['distinct_home_lineups']

player_encoding_away = player_outcome_away.merge(
    opp_stats_away[['season', 'team', 'player', 'avg_opp_occurrence']],
    on=['season', 'team', 'player'], how='left'
)
player_encoding_away['avg_opp_occurrence'] = player_encoding_away['avg_opp_occurrence'].fillna(0)
player_encoding_away['final_mean_encoding'] = player_encoding_away['base_mean_encoding'] * (1 + lambda_param * player_encoding_away['avg_opp_occurrence'])

mean_encoding_dict_away = dict(zip(
    zip(player_encoding_away['season'], player_encoding_away['team'], player_encoding_away['player']),
    player_encoding_away['final_mean_encoding']
))



In [35]:
player_encoding_home.head()

,season,team,player,total_weighted_outcome,total_minutes,base_mean_encoding,avg_opp_occurrence,final_mean_encoding
0,2007,ATL,Anthony Johnson,-69.0,449.0,-0.153675,1.511450,-0.165288
1,2007,ATL,Cedric Bozeman,-33.0,75.0,-0.440000,1.228571,-0.467029
2,2007,ATL,Dijon Thompson,-4.0,22.0,-0.181818,1.181818,-0.192562
3,2007,ATL,Esteban Batista,0.0,8.0,0.000000,1.000000,0.000000
4,2007,ATL,Jeremy Richardson,-4.0,14.0,-0.285714,1.375000,-0.305357


In [25]:
import plotly.express as px

fig = px.scatter(
    player_encoding_home,
    x="total_minutes",
    y="final_mean_encoding",
    hover_data=["player", "team", "season"],
    title="Relation between Final Mean Encoding and Total Minutes Played (Home Players)"
)
fig.update_layout(
    xaxis_title="Total Minutes Played",
    yaxis_title="Final Mean Encoding"
)
import plotly.io as pio
pio.renderers.default = 'browser'
fig.show()



In [15]:
# =============================================================================
# STEP 4: Apply Mean Encoding to Raw Data to Create df_encoded
# =============================================================================
df_encoded = df_filtered.copy()

# For home players, when applying the encoding, also multiply by the position frequency.
for col in ['home_0', 'home_1', 'home_2', 'home_3', 'home_4']:
    # Extract the position from the column name (e.g., "home_0" -> "home_0")
    pos = col
    df_encoded[col] = df_encoded.apply(lambda row: 
        mean_encoding_dict_home.get((row['season'], row['home_team'], row[col]), np.nan) *
        pos_freq_dict_home.get((row['season'], row['home_team'], row[col], pos), 0),
        axis=1)
df_encoded[['home_0', 'home_1', 'home_2', 'home_3', 'home_4']] = df_encoded[['home_0', 'home_1', 'home_2', 'home_3', 'home_4']].fillna(df_encoded['outcome'].mean())

# For away players, do similarly.
for col in ['away_0', 'away_1', 'away_2', 'away_3', 'away_4']:
    pos = col
    df_encoded[col] = df_encoded.apply(lambda row:
        mean_encoding_dict_away.get((row['season'], row['away_team'], row[col]), np.nan) *
        pos_freq_dict_away.get((row['season'], row['away_team'], row[col], pos), 0),
        axis=1)
df_encoded[['away_0', 'away_1', 'away_2', 'away_3', 'away_4']] = df_encoded[['away_0', 'away_1', 'away_2', 'away_3', 'away_4']].fillna(df_encoded['outcome'].mean())

# Drop the lineup columns since they will not be used for training.
df_encoded = df_encoded.drop(columns=['date', 'home_lineup', 'away_lineup'])

print("Sample of df_encoded with Updated Mean Encodings:")
print(df_encoded[['season', 'home_team', 'home_0', 'home_1', 'home_2', 'home_3', 'home_4',
                   'away_team', 'away_0', 'away_1', 'away_2', 'away_3', 'away_4']].head())




Sample of df_encoded with Updated Mean Encodings:
   season home_team    home_0    home_1    home_2    home_3    home_4  \
0    2007       LAL -0.140845 -0.033028 -0.050393 -0.015363 -0.126940   
1    2007       LAL -0.140845 -0.033028 -0.050393 -0.015363 -0.126940   
2    2007       LAL -0.001651 -0.009664 -0.025935 -0.059052 -0.126940   
3    2007       LAL -0.001651 -0.009664 -0.025935 -0.059052 -0.126940   
4    2007       LAL -0.000345 -0.006587 -0.025255 -0.021419 -0.226451   

  away_team    away_0    away_1    away_2    away_3    away_4  
0       PHO  0.275351  0.255578  0.299529  0.412070  0.559661  
1       PHO  0.557832  0.170317  0.299529  0.412070  0.559661  
2       PHO  0.557832  0.170317  0.299529  0.412070  0.559661  
3       PHO  0.275351  0.323751  0.160387  0.130233  0.067705  
4       PHO  0.275351  0.323751  0.160387  0.130233  0.067705  


In [37]:
df_encoded.to_csv("../data/encoded_data1.csv", index=False)

In [38]:
# =============================================================================
# STEP 5: Create Training Data by Simulating Missingness
# =============================================================================
training_rows = []
for idx, enc_row in df_encoded.iterrows():
    raw_row = df_filtered.loc[idx]  # Using raw data for true player names
    for pos in range(5):
        feature_row = enc_row.copy()
        feature_row['missing_position'] = pos
        feature_row['missing_player'] = raw_row[f'home_{pos}']
        # Instead of dropping the column, we set the value to NaN to simulate missingness.
        feature_row[f'home_{pos}'] = np.nan
        training_rows.append(feature_row)

df_train = pd.DataFrame(training_rows).reset_index(drop=True)


In [40]:
df_train.to_csv("../data/train.csv", index=False)

In [ ]:
df_train = pd.DataFrame(training_rows).reset_index(drop=True)



In [3]:
df_train = pd.read_csv("../data/train.csv")

In [41]:
df_train = df_train.drop(columns=['date'])

KeyError: "['date'] not found in axis"

In [4]:
df_train

,game,season,home_team,away_team,starting_min,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4,outcome,missing_position,missing_player
0,200610310LAL,2007,LAL,PHO,0,NaN,-0.033028,-0.050393,-0.015363,-0.126940,0.275351,0.255578,0.299529,0.412070,0.559661,-1,0,Andrew Bynum
1,200610310LAL,2007,LAL,PHO,0,-0.140845,NaN,-0.050393,-0.015363,-0.126940,0.275351,0.255578,0.299529,0.412070,0.559661,-1,1,Lamar Odom
2,200610310LAL,2007,LAL,PHO,0,-0.140845,-0.033028,NaN,-0.015363,-0.126940,0.275351,0.255578,0.299529,0.412070,0.559661,-1,2,Luke Walton
3,200610310LAL,2007,LAL,PHO,0,-0.140845,-0.033028,-0.050393,NaN,-0.126940,0.275351,0.255578,0.299529,0.412070,0.559661,-1,3,Sasha Vujacic
4,200610310LAL,2007,LAL,PHO,0,-0.140845,-0.033028,-0.050393,-0.015363,NaN,0.275351,0.255578,0.299529,0.412070,0.559661,-1,4,Smush Parker
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1184555,201504050NYK,2015,NYK,PHI,45,NaN,-0.064095,-0.126003,-0.037864,-0.168152,0.248653,-0.018113,0.054061,-0.155025,-0.138758,1,0,Andrea Bargnani
1184556,201504050NYK,2015,NYK,PHI,45,-0.250780,NaN,-0.126003,-0.037864,-0.168152,0.248653,-0.018113,0.054061,-0.155025,-0.138758,1,1,Jason Smith
1184557,201504050NYK,2015,NYK,PHI,45,-0.250780,-0.064095,NaN,-0.037864,-0.168152,0.248653,-0.018113,0.054061,-0.155025,-0.138758,1,2,Lance Thomas
1184558,201504050NYK,2015,NYK,PHI,45,-0.250780,-0.064095,-0.126003,NaN,-0.168152,0.248653,-0.018113,0.054061,-0.155025,-0.138758,1,3,Langston Galloway


In [5]:
# =============================================================================
# STEP 6: Reduce Training Set by 75%
# =============================================================================
df_train = df_train.sample(frac=0.25, random_state=42).reset_index(drop=True)
df_train

,game,season,home_team,away_team,starting_min,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4,outcome,missing_position,missing_player
0,200703170SEA,2007,SEA,GSW,12,-0.109869,-0.383730,-0.532633,NaN,-0.607064,0.483599,0.144623,0.214252,0.167333,0.207589,-1,3,Nick Collison
1,201001180NOH,2010,NOH,SAS,25,-0.939921,NaN,-0.856578,-0.370859,-1.060434,0.456236,0.334585,0.161894,0.242605,0.516395,-1,1,David West
2,201001180CHA,2010,CHA,SAC,33,NaN,-0.541094,-0.301271,-0.046634,-0.564009,0.205176,0.075013,0.125186,0.127032,0.186402,-1,0,Boris Diaw
3,201504050CLE,2015,CLE,CHI,33,0.019039,-0.032997,-0.032348,NaN,-0.118809,0.380171,0.237199,0.148192,0.139489,0.264839,1,3,LeBron James
4,201303180PHI,2013,PHI,POR,37,NaN,-0.036458,-0.013189,-0.001187,-0.039809,0.255518,0.267369,0.097975,0.011798,0.188222,-1,0,Charles Jenkins
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296135,201503070NYK,2015,NYK,IND,31,-0.181949,-0.117476,-0.074517,NaN,-0.207718,0.246085,0.090237,0.134792,0.144021,0.138855,-1,3,Shane Larkin
296136,200611270MIA,2007,MIA,PHI,10,-0.137593,-0.194308,-0.072585,NaN,-0.249376,0.526248,0.162085,0.221200,0.221832,0.458591,-1,3,Jason Kapono
296137,200911070MIL,2010,MIL,NYK,29,-0.092423,NaN,-0.052438,-0.089843,-0.123580,0.292909,0.082477,0.133192,0.342973,0.479003,-1,1,Hakim Warrick
296138,200903030ORL,2009,ORL,PHO,9,-0.081600,-0.018702,-0.025167,NaN,-0.050981,0.023082,0.076941,0.097186,0.066304,0.034353,-1,3,Rafer Alston


In [16]:
# =============================================================================
# STEP 7: Define Features (X) and Target (y)
# =============================================================================
y = df_train['missing_player'].copy()
X = df_train.drop(columns=['game', 'missing_player'])

In [17]:
# =============================================================================
# STEP 8: Process Categorical Features
# =============================================================================
# We'll use 'season', 'home_team', and 'away_team'. Note that these column names come from df_encoded.
categorical_cols = ['season', 'home_team', 'away_team']
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# =============================================================================
# STEP 9: Encode the Target Variable
# =============================================================================
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Target Classes:", le.classes_)

Target Classes: ['A.J. Price' 'Aaron Brooks' 'Aaron Gordon' ... 'Zaza Pachulia'
 'Zoran Dragic' 'Zydrunas Ilgauskas']


In [18]:
# =============================================================================
# STEP 10: Split Data into Training and Test Sets
# =============================================================================
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.20, random_state=42)

# =============================================================================
# STEP 11: Train a Random Forest Classifier
# =============================================================================
clf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)


RandomForestClassifier(max_depth=20, n_jobs=-1, random_state=42)

In [43]:
# =============================================================================
# STEP 12: Evaluate the Model
# =============================================================================
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

# Only report the classes that are present in y_test.
import numpy as np
unique_labels = np.unique(y_test)
target_names = [le.classes_[i] for i in unique_labels]

print("\nClassification Report:")
print(classification_report(y_test, y_pred, labels=unique_labels, target_names=target_names))

Accuracy: 0.5246842709529277

Classification Report:
                          precision    recall  f1-score   support

              A.J. Price       0.80      0.44      0.57        27
            Aaron Brooks       0.54      0.85      0.66        67
            Aaron Gordon       0.43      0.43      0.43         7
              Aaron Gray       1.00      0.22      0.36        18
             Aaron McKie       0.00      0.00      0.00         1
          Aaron Williams       0.00      0.00      0.00        10
                Acie Law       0.50      0.08      0.13        13
           Adam Morrison       0.60      0.68      0.64        22
            Adonal Foyle       0.00      0.00      0.00         9
           Adreian Payne       0.00      0.00      0.00         4
          Adrian Griffin       0.00      0.00      0.00         5
           Al Harrington       0.64      0.80      0.71        88
              Al Horford       0.40      0.94      0.56        80
            Al Jeffers

In [45]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# For reproducibility
np.random.seed(42)

# =============================================================================
# Create Test Data with a Random Missing Home Position
# =============================================================================

# 1. Filter the original data for games where outcome==1 (home win)
df_test_filtered = df_filtered[df_filtered['outcome'] == 1].copy()

# 2. Get corresponding rows from the mean-encoded data (df_encoded)
df_test = df_encoded.loc[df_test_filtered.index].copy()

# 3. For each test row, choose a random home player position to be missing.
def simulate_random_missing(row_enc, row_raw):
    pos = np.random.choice(range(5))
    row_enc['missing_position'] = pos
    row_enc['missing_player'] = row_raw[f'home_{pos}']  # Save the true player name
    row_enc[f'home_{pos}'] = np.nan  # Simulate missingness by setting it to NaN
    return row_enc

df_test = df_test.apply(lambda row: simulate_random_missing(row, df_test_filtered.loc[row.name]), axis=1)

# Optional: Display a sample of the test data
print("Test Data Sample with Random Missing Home Position:")
print(df_test[['season', 'home_team', 'home_0', 'home_1', 'home_2', 'home_3', 'home_4',
               'away_team', 'away_0', 'away_1', 'away_2', 'away_3', 'away_4', 
               'missing_position', 'missing_player']].head())

# =============================================================================
# Process Test Data Features to Match Training Data
# =============================================================================
# Process categorical features as in training (assuming training used one-hot encoding on 'season', 'home_team', 'away_team')
categorical_cols = ['season', 'home_team', 'away_team']
df_test_processed = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

# Reindex the test DataFrame to match the training columns (X_train) from your training pipeline.
# This will add any missing columns (filled with 0) that might have been present in training.
df_test_processed = df_test_processed.reindex(columns=X_train.columns, fill_value=0)


Test Data Sample with Random Missing Home Position:
   season home_team    home_0    home_1    home_2    home_3    home_4  \
2    2007       LAL -0.001651 -0.009664 -0.025935       NaN -0.126940   
3    2007       LAL -0.001651 -0.009664 -0.025935 -0.059052       NaN   
5    2007       LAL  0.030370 -0.006587       NaN -0.021419 -0.226451   
6    2007       LAL  0.030370 -0.056291 -0.092754 -0.015363       NaN   
7    2007       LAL  0.030370 -0.056291 -0.092754 -0.015363       NaN   

  away_team    away_0    away_1    away_2    away_3    away_4  \
2       PHO  0.557832  0.170317  0.299529  0.412070  0.559661   
3       PHO  0.275351  0.323751  0.160387  0.130233  0.067705   
5       PHO  0.275351  0.323751  0.160387  0.130233  0.067705   
6       PHO  0.275351  0.323751  0.160387  0.130233  0.067705   
7       PHO  0.275351  0.323751  0.141265  0.412070  0.559661   

   missing_position       missing_player  
2                 3         Ronny Turiaf  
3                 4         Smus

In [ ]:

# =============================================================================
# Evaluate the Model on Test Data
# =============================================================================
# Predict using your trained classifier.
y_test_pred = clf.predict(df_test_processed)

# Encode the true missing player using the same LabelEncoder (le) that was fit on the training data.
# If a label in the test set was unseen during training, we filter to only use known labels.
y_test_true = le.transform(df_test['missing_player'])

# To avoid errors if some classes in the encoder are not present in y_test, extract only the unique labels present in y_test.
unique_labels = np.unique(y_test_true)
target_names = [le.classes_[i] for i in unique_labels]

print("Test Accuracy:", accuracy_score(y_test_true, y_test_pred))
print("\nTest Classification Report:")
print(classification_report(y_test_true, y_test_pred, labels=unique_labels, target_names=target_names))

In [ ]:
import numpy as np

def top_k_accuracy_score(y_true, y_pred_proba, k=3):
    """
    Computes the top-K accuracy.
    
    Parameters:
        y_true (array-like): True class indices.
        y_pred_proba (ndarray): Predicted probability array (n_samples x n_classes).
        k (int): Number of top predictions to consider.
        
    Returns:
        float: Top-K accuracy.
    """
    # Get the indices of the top k predictions for each sample.
    top_k_preds = np.argsort(y_pred_proba, axis=1)[:, -k:]
    
    # Check whether the true label is in the top k predictions for each sample.
    correct = [y_true[i] in top_k_preds[i] for i in range(len(y_true))]
    return np.mean(correct)

# After training your Random Forest model (clf), get predicted probabilities on the test set.
y_pred_proba = clf.predict_proba(X_test)

# Calculate Top-3 accuracy.
top_5_acc = top_k_accuracy_score(y_test, y_pred_proba, k=5)
print("Top-3 Accuracy:", top_3_acc)


Top-3 Accuracy: 0.18774903761734316


In [43]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# ----------------------------
# Load Test Data
# ----------------------------
df_test = pd.read_csv("../data/NBA_test.csv")
df_test_labels = pd.read_csv("../data/NBA_test_labels.csv")



In [83]:
import pandas as pd
import numpy as np

# -------------------------------
# Load Test Data and Labels
# -------------------------------
# Load the test data (NBA_test.csv) and the test labels (NBA_test_labels.csv)
df_test = pd.read_csv("NBA_test.csv")
df_test_labels = pd.read_csv("NBA_test_labels.csv")  # assumed to have a column 'missing_player'

# Merge the missing player labels into the test dataframe (assuming they align by index)
df_test['missing_player'] = df_test_labels['missing_player']

# -------------------------------
# Clean Home Player Columns in Test Data
# -------------------------------
# Replace "?" with NaN in the home player columns.
home_cols = ['home_0', 'home_1', 'home_2', 'home_3', 'home_4']
for col in home_cols:
    df_test[col] = df_test[col].replace("?", np.nan)

# -------------------------------
# Determine the Missing Home Position
# -------------------------------
def find_missing_position(row):
    for i, col in enumerate(home_cols):
        if pd.isna(row[col]):
            return i
    return np.nan

df_test['missing_position'] = df_test.apply(find_missing_position, axis=1)

# -------------------------------
# Define Fallback and Lookup Functions for Mean Encoding
# -------------------------------
# Use overall_mean_outcome from training as fallback if not available.
fallback_value = 'overall_mean_outcome' in globals() else 0.5

def get_home_mean_encoding(season, team, player, position):
    # If player is missing, return NaN.
    if pd.isna(player):
        return np.nan
    key = (season, team, player)
    if key in mean_encoding_dict_home:
        return mean_encoding_dict_home[key]
    # For season 2016, try to get the most recent available encoding.
    if season == 2016:
        possible = [k for k in mean_encoding_dict_home.keys() if k[1] == team and k[2] == player and k[0] < 2016]
        if possible:
            best_key = max(possible, key=lambda x: x[0])
            return mean_encoding_dict_home[best_key]
    return fallback_value

def get_away_mean_encoding(season, team, player, position):
    if pd.isna(player):
        return np.nan
    key = (season, team, player)
    if key in mean_encoding_dict_away:
        return mean_encoding_dict_away[key]
    if season == 2016:
        possible = [k for k in mean_encoding_dict_away.keys() if k[1] == team and k[2] == player and k[0] < 2016]
        if possible:
            best_key = max(possible, key=lambda x: x[0])
            return mean_encoding_dict_away[best_key]
    return fallback_value

# -------------------------------
# Replace Player Names with Mean Encoded Values in Test Data
# -------------------------------
# Process Home Players:
for col in home_cols:
    # For each row, replace the player value with the mean encoding
    df_test[col] = df_test.apply(lambda row: get_home_mean_encoding(row['season'], row['home_team'], row[col], col), axis=1)

# Process Away Players:
away_cols = ['away_0', 'away_1', 'away_2', 'away_3', 'away_4']
for col in away_cols:
    df_test[col] = df_test.apply(lambda row: get_away_mean_encoding(row['season'], row['away_team'], row[col], col), axis=1)

# -------------------------------
# Finalize Test Data
# -------------------------------
# Ensure the test dataframe contains the desired columns:
desired_columns = [
    'season',
    'home_team',
    'away_team',
    'starting_min',
    'home_0',
    'home_1',
    'home_2',
    'home_3',
    'home_4',
    'away_0',
    'away_1',
    'away_2',
    'away_3',
    'away_4',
    'outcome',
    'missing_position',
    'missing_player'
]

df_test_final = df_test[desired_columns].copy()

print("Final Test Data Sample:")
print(df_test_final.head())


# 🔹 Replace "CHO" with "CHA" in the home_team column
df_test_encoded['home_team'] = df_test_encoded['home_team'].replace('CHO', 'CHA')
df_test_encoded['away_team'] = df_test_encoded['away_team'].replace('CHO', 'CHA')



SyntaxError: invalid syntax (872577894.py, line 37)

In [77]:
df_test_encoded

,season,home_team,away_team,starting_min,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4,outcome,missing_position,missing_player
0,2007,IND,BOS,18,-0.211653,-0.118712,-0.050468,-0.175366,NaN,0.225721,0.226988,0.239045,0.160292,0.329863,1,-1,Troy Murphy
1,2007,HOU,DAL,16,-0.176321,NaN,-0.021138,-0.029487,0.016117,0.363762,0.131087,0.150475,0.517173,0.790072,1,-1,Chuck Hayes
2,2007,SAS,POR,39,-0.266576,NaN,-0.001545,-0.064483,-0.004597,0.474899,0.197807,0.177786,0.105840,0.432677,1,-1,Brent Barry
3,2007,MIN,BOS,21,NaN,-0.052404,-0.100486,-0.114122,-0.179673,0.558692,0.253923,0.137731,0.250902,0.156585,1,-1,Craig Smith
4,2007,MEM,LAL,19,-0.183680,-0.121543,-0.064296,-0.100517,NaN,0.412166,0.260634,0.221375,0.229887,0.250755,1,-1,Stromile Swift
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2016,BRK,CHA,20,NaN,-0.000000,-0.000000,-0.000000,-0.000000,0.000000,-0.000000,-0.000000,0.000000,-0.000000,1,-1,Donald Sloan
996,2016,TOR,CHA,21,-0.000000,-0.000000,NaN,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.000000,-0.000000,1,-1,Jonas Valanciunas
997,2016,DAL,DET,13,-0.000000,NaN,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,1,-1,Devin Harris
998,2016,NOP,ORL,24,-0.000000,-0.000000,NaN,-0.000000,-0.000000,-0.000000,0.000000,-0.000000,0.000000,0.000000,1,-1,Ish Smith


In [67]:
# Define the desired column order for the test set.
desired_columns = [
    'season',
    'home_team',
    'away_team',
    'starting_min',
    'home_0',
    'home_1',
    'home_2',
    'home_3',
    'home_4',
    'away_0',
    'away_1',
    'away_2',
    'away_3',
    'away_4',
    'outcome',
    'missing_position',
    'missing_player'
]

# If your processed test DataFrame (df_test_final) has extra columns (like dummy variables),
# reindex to keep only the desired columns. Missing columns will be filled with zeros.
df_test_final = df_test_encoded.reindex(columns=desired_columns)

# Optionally, you can reorder columns explicitly (if reindex doesn't preserve order):
df_test_final = df_test_final[desired_columns]

print("Final Test Data for Prediction:")
display(df_test_final.head())


Final Test Data for Prediction:


,season,home_team,away_team,starting_min,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4,outcome,missing_position,missing_player
0,2007,IND,BOS,18,-0.271711,-0.186761,-0.080375,-0.278219,-0.255631,0.404146,0.545216,0.504930,0.438136,0.593511,1,NaN,Troy Murphy
1,2007,HOU,DAL,16,-0.176321,-0.255631,-0.123594,-0.176302,0.025264,0.465084,0.740504,0.399565,0.782552,0.809052,1,NaN,Chuck Hayes
2,2007,SAS,POR,39,-0.266576,-0.255631,-0.032610,-0.192297,-0.022249,0.474899,0.414377,0.447051,0.400810,0.432677,1,NaN,Brent Barry
3,2007,MIN,BOS,21,-0.255631,-0.147462,-0.231119,-0.198835,-0.192476,0.558692,0.446070,0.496730,0.646629,0.438136,1,NaN,Craig Smith
4,2007,MEM,LAL,19,-0.248830,-0.315404,-0.169213,-0.299798,-0.255631,0.423903,0.535687,0.394888,0.449169,0.343300,1,NaN,Stromile Swift
